In [4]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("aslak")

rho = 300
rhoi = 917
rhoterm = rhoi - rho
DrhoDt = 30
dt = 7

print("plain", rho + DrhoDt * dt)
# old based on herron langway style: rho' = c*(rhoi-rho)
print("+0", rhoi - rhoterm * np.exp(-DrhoDt * dt / (rhoterm)))
print("+1", rhoi - rhoterm * np.exp(-DrhoDt * dt / (rhoterm + 1)))
print("+10", rhoi - rhoterm * np.exp(-DrhoDt * dt / (rhoterm + 10)))

# new based on integrating rho' = c*(rhoi-rho)*rho
print("newx", -(rho * rhoi) / (np.exp(-(-DrhoDt * rhoi * dt) / (rho * (rho - rhoi))) * (rho - rhoi) - rho))
print("newx+1", -(rho * rhoi) / (np.exp(-(-DrhoDt * rhoi * dt) / (rho * (rho - rhoi - 1))) * (rho - rhoi) - rho))

plain 510
+0 477.9942724127098
+1 477.7524283449805
+10 475.6047227703702
newx 531.0771927347182
newx+1 530.7008874977043


In [ ]:
import numpy as np
from numba import njit
from scipy.interpolate import make_interp_spline


@njit
def find_span(t, k, x):
    n = len(t) - k - 1

    if x >= t[n]:
        return n - 1
    if x <= t[k]:
        return k

    low = k
    high = n
    mid = (low + high) // 2

    while x < t[mid] or x >= t[mid + 1]:
        if x < t[mid]:
            high = mid
        else:
            low = mid
        mid = (low + high) // 2

    return mid


@njit
def de_boor_single(x, t, c, k):
    i = find_span(t, k, x)

    d = np.empty(k + 1)
    for j in range(k + 1):
        d[j] = c[i - k + j]

    for r in range(1, k + 1):
        for j in range(k, r - 1, -1):
            left = t[i - k + j]
            right = t[i + 1 + j - r]

            if right - left == 0.0:
                alpha = 0.0
            else:
                alpha = (x - left) / (right - left)

            d[j] = (1.0 - alpha) * d[j - 1] + alpha * d[j]

    return d[k]


@njit
def spline_eval(x_vals, t, c, k):
    n = len(x_vals)
    out = np.empty(n)

    for idx in range(n):
        out[idx] = de_boor_single(x_vals[idx], t, c, k)

    return out


x = np.sort(np.random.randn(50))
y = np.cos(x) + np.random.randn(len(x))

spl = make_interp_spline(x, y, 3)

de_boor_single(0, spl.t, spl.c, spl.k)

10.250407948577468